# Clinical Burden Monitoring Results

This notebook summarizes the two clinical layers added to the breast FTV rollout paper: low/high residual MRI-burden probabilities from Monte Carlo samples, and serial monitoring behavior as additional MRI visits are conditioned on.

## Layer 1: Low and High Residual MRI Burden

The clinical readout is framed around MRI burden rather than pathology response. The main low-burden endpoint is final visit `T3 FTV < 5 mL`; the high-burden endpoint is `T3 FTV > 20 mL`. The retained Hybrid-Edge k=8 model produces Monte Carlo probability scores for both thresholds. Last-observed FTV is included as a clinical-use baseline because current FTV is already highly informative.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path('/Users/irisseaman/Research/3DGCNN')
threshold = pd.read_csv(ROOT / 'reports/bio_ftv_clinical_burden_monitoring/clinical_burden_threshold_readouts.csv')
monitor = pd.read_csv(ROOT / 'reports/bio_ftv_clinical_burden_monitoring/clinical_burden_monitoring_by_visit.csv')
threshold

Key interpretation: the retained MC readout strongly identifies low residual burden (AUC 0.957) and high residual burden (AUC 0.984). The last-observed FTV baseline is similarly strong, so the paper should not claim that the graph is needed just to threshold current FTV. The contribution is that the graph model turns the burden forecast into a patient-level probability distribution while preserving spatial graph-state outputs.

## Layer 2: Serial Monitoring Across Visits

The second layer asks whether the readout remains useful when the forecast is updated after later MRI visits. This is a monitoring interpretation rather than a treatment-selection claim.

In [ ]:
monitor

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
x = range(len(monitor))
axes[0].plot(x, monitor['auc_t3_ftv_lt5'], marker='o', label='T3 FTV < 5 mL')
axes[0].plot(x, monitor['auc_t3_ftv_gt20'], marker='s', label='T3 FTV > 20 mL')
axes[0].set_xticks(list(x), monitor['conditioning'])
axes[0].set_ylim(0.90, 1.00)
axes[0].set_ylabel('AUC')
axes[0].set_title('Burden-threshold discrimination')
axes[0].legend(frameon=False)
axes[1].bar([i-0.18 for i in x], monitor['mc_mean_ftv_mae_ml'], width=0.36, label='MC-mean MAE')
axes[1].bar([i+0.18 for i in x], monitor['raw_90_width_ml'], width=0.36, label='Raw 90% width')
axes[1].set_xticks(list(x), monitor['conditioning'])
axes[1].set_ylabel('mL')
axes[1].set_title('Error and interval width')
axes[1].legend(frameon=False)
for ax in axes:
    ax.spines[['top','right']].set_visible(False)
    ax.grid(axis='y', alpha=0.3)
plt.tight_layout()

The low- and high-burden discrimination remains high across T0-only, T0+T1, and T0+T1+T2 conditioning. Endpoint error and interval width do not shrink monotonically under the current residual-pool design. The supported clinical language is therefore serial uncertainty-aware MRI burden monitoring, not guaranteed increasing confidence after every scan.

## Paper-facing conclusion

The strongest clinical phrasing is: the framework estimates the probability of low or substantial residual enhancing tumor burden at the final treatment MRI while preserving graph-native tumor-state forecasts. pCR should remain exploratory because the model was trained to forecast imaging burden, not pathology response.